# Notebook 10: Data Poisoning

## Why This Matters
Backdoor attacks (notebook 08) install hidden conditional behaviors. Data poisoning attacks are broader: they aim to degrade the model's general accuracy, cause targeted misclassification without any trigger, or force the model to learn wrong decision boundaries. Any time your training data comes from an untrusted source — crowdsourced labels, web-scraped datasets, federated learning participants — you're exposed.

## Structure
1. Poisoning taxonomy: availability vs integrity attacks
2. Label flipping: the simplest attack
3. Gradient-based poisoning: optimal poisoning with model access
4. Influence functions: understanding which training examples matter most
5. Clean-label targeted poisoning (Witches' Brew, Geiping et al.)
6. Defenses: data sanitization and robust training
7. `poison-control` prototype: a training set scanner

## What You'll Learn
- Why random label flipping is surprisingly effective at low rates
- How influence functions identify the most impactful training examples
- Why clean-label attacks are more dangerous than label-flipping attacks
- How to detect poisoning through loss distribution analysis and clustering

## Background

### Availability vs integrity attacks

Data poisoning attacks fall into two categories based on what the attacker wants:

**Availability attacks**: degrade the model's overall accuracy — make it useless. Simple label flipping at scale achieves this. The attacker doesn't care about any specific misclassification; they just want the model to fail. This matters in competitive contexts (sabotaging a competitor's model) or in adversarial data collection scenarios.

**Integrity attacks**: cause specific targeted misclassification without degrading general accuracy. These are more sophisticated — the attacker wants the model to appear functional while having a specific exploitable flaw. Targeted poisoning (this notebook) and backdoor attacks (notebook 08) are both integrity attacks. The distinction from backdoors: integrity attacks don't require a trigger; the attack target is a specific test instance or class.

### Label flipping: Biggio et al. (2012)

The earliest formal treatment of data poisoning in modern ML is Biggio, Nelson & Laskov (2012) — *"Poisoning Attacks against Support Vector Machines"* — which showed that an attacker who can inject malicious training examples can reliably degrade classifier accuracy. This was originally demonstrated for SVMs, where the optimal poisoning point lies on the decision boundary.

For neural networks, even naive label flipping (randomly change some labels to wrong classes) is effective. At 10% flip rate, accuracy typically degrades by 5–15% depending on the task and model size. At 30% flip rate, models often fail to learn meaningful decision boundaries.

### Gradient-based poisoning: Munoz-Gonzalez et al. (2017)

Optimal poisoning — finding the training examples that, when perturbed, cause maximum damage — is a bilevel optimization problem:

$$\max_D \mathcal{L}_{\text{test}}(\theta^*(D))  \quad \text{s.t.} \quad \theta^* = \arg\min_\theta \mathcal{L}_{\text{train}}(\theta, D)$$

The outer maximization finds the poisoned dataset $D$ that maximizes test loss after the inner minimization (training) finds optimal weights $\theta^*$. Computing the exact gradient of the outer objective requires differentiating through the training process — which is possible via implicit differentiation and influence functions.

### Influence functions: understanding training data impact

Koh & Liang (2017) — *"Understanding Black-box Predictions via Influence Functions"* — formalized a tool from robust statistics for understanding which training examples most influence model predictions. The influence of training point $(x_i, y_i)$ on the loss at test point $(x_t, y_t)$ is:

$$\mathcal{I}(x_i, x_t) = -\nabla_\theta \mathcal{L}(x_t)^T H_\theta^{-1} \nabla_\theta \mathcal{L}(x_i)$$

where $H_\theta$ is the Hessian of the training loss. Positive influence means the training point helps predict the test point correctly; negative influence means it hurts. Influence functions are the right tool for finding which training examples an adversary should corrupt to cause targeted misclassification.

### Witches' Brew: targeted clean-label poisoning (Geiping et al., 2021)

Witches' Brew (Geiping, Fowl, Huang, Czaja, Taylor, Goldblum & Goldstein, 2021) introduced a clean-label targeted attack that's more effective than prior work: by crafting a *brew* of poisoned examples (not just one) that all push the model toward misclassifying a specific target instance, and doing so while keeping all labels correct, the attack is both effective and nearly undetectable through label inspection.

In [ ]:
%pip install -q torch torchvision numpy matplotlib scikit-learn

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest

torch.manual_seed(42)
np.random.seed(42)
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

device = torch.device('mps' if torch.backends.mps.is_available() else
                      'cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

transform = transforms.Compose([
    transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))
])
train_data = torchvision.datasets.MNIST('./data', train=True,  download=True, transform=transform)
test_data  = torchvision.datasets.MNIST('./data', train=False, download=True, transform=transform)
test_loader = DataLoader(test_data, batch_size=256, shuffle=False)

In [ ]:
class MnistCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool  = nn.MaxPool2d(2)
        self.fc1   = nn.Linear(64 * 7 * 7, 128)
        self.fc2   = nn.Linear(128, 10)
        self.drop  = nn.Dropout(0.25)
    def forward(self, x):
        x = F.relu(self.conv1(x)); x = self.pool(x)
        x = F.relu(self.conv2(x)); x = self.pool(x)
        x = F.relu(self.fc1(x.flatten(1))); x = self.drop(x)
        return self.fc2(x)


def evaluate(model, loader, device, n_batches=None):
    model.eval(); correct = total = 0
    with torch.no_grad():
        for i, (x, y) in enumerate(loader):
            if n_batches and i >= n_batches: break
            x, y = x.to(device), y.to(device)
            correct += (model(x).argmax(1) == y).sum().item(); total += y.size(0)
    return correct / total


def train_model(loader, epochs=8, seed=42):
    torch.manual_seed(seed)
    m = MnistCNN().to(device)
    o = torch.optim.Adam(m.parameters(), lr=1e-3)
    for _ in range(epochs):
        m.train()
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            o.zero_grad(); F.cross_entropy(m(x), y).backward(); o.step()
    return m

## 1. Label Flipping Attack

In [ ]:
class LabelFlippedDataset(Dataset):
    """
    Randomly flip labels of a fraction of the training set.
    Optionally: targeted flipping (class A → class B only).
    """
    def __init__(self, base_dataset, flip_rate: float, source_class=None, target_class=None, seed=42):
        self.base = base_dataset
        rng = np.random.RandomState(seed)

        all_labels = np.array([base_dataset[i][1] for i in range(len(base_dataset))])

        if source_class is not None:
            # Targeted: only flip source_class → target_class
            eligible = np.where(all_labels == source_class)[0]
        else:
            eligible = np.arange(len(base_dataset))

        n_flip = int(len(eligible) * flip_rate)
        flip_indices = set(rng.choice(eligible, n_flip, replace=False).tolist())

        self.new_labels = all_labels.copy()
        for idx in flip_indices:
            if source_class is not None and target_class is not None:
                self.new_labels[idx] = target_class
            else:
                # Random flip to any other class
                wrong_classes = [c for c in range(10) if c != all_labels[idx]]
                self.new_labels[idx] = rng.choice(wrong_classes)

        self.n_flipped = len(flip_indices)

    def __len__(self): return len(self.base)
    def __getitem__(self, idx):
        x, _ = self.base[idx]
        return x, int(self.new_labels[idx])


# Measure accuracy degradation as a function of flip rate
flip_rates = [0.0, 0.01, 0.05, 0.10, 0.20, 0.30, 0.50]
print("Availability attack: random label flipping")
print(f"{'Flip rate':>12} {'Test accuracy':>15} {'Accuracy drop':>15}")
print("-" * 45)

base_acc = None
accuracies = []
for rate in flip_rates:
    if rate == 0:
        ds = train_data
    else:
        ds = LabelFlippedDataset(train_data, flip_rate=rate)
    loader = DataLoader(ds, batch_size=256, shuffle=True)
    model  = train_model(loader, epochs=6)
    acc    = evaluate(model, test_loader, device, n_batches=30)
    if base_acc is None: base_acc = acc
    accuracies.append(acc)
    print(f"{rate:>12.2f} {acc:>15.4f} {acc - base_acc:>+15.4f}")

In [ ]:
# Targeted label flipping: 7 → 1 (visually similar)
print("\nIntegrity attack: targeted label flipping (7 → 1)")
ds_targeted = LabelFlippedDataset(train_data, flip_rate=0.5, source_class=7, target_class=1)
loader_targeted = DataLoader(ds_targeted, batch_size=256, shuffle=True)
model_targeted = train_model(loader_targeted, epochs=8)

# Evaluate: overall accuracy vs accuracy on class 7
model_targeted.eval()
correct_all = correct_7 = total_all = total_7 = 0
with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        preds = model_targeted(x).argmax(1)
        correct_all += (preds == y).sum().item(); total_all += y.size(0)
        mask7 = y == 7
        if mask7.sum() > 0:
            correct_7 += (preds[mask7] == y[mask7]).sum().item()
            total_7   += mask7.sum().item()

print(f"  Overall test accuracy: {correct_all/total_all:.4f}")
print(f"  Class 7 accuracy:      {correct_7/total_7:.4f}  ← targeted attack")
print(f"  (50% of 7s relabeled as 1 in training)")

## 2. Influence Functions: Finding High-Impact Training Examples

In [ ]:
def compute_grad(model, x, y, device):
    """Compute gradient of loss w.r.t. model parameters for a single example."""
    model.eval()
    x, y = x.unsqueeze(0).to(device), torch.tensor([y], device=device)
    loss = F.cross_entropy(model(x), y)
    grads = torch.autograd.grad(loss, model.parameters(), create_graph=False)
    return torch.cat([g.flatten() for g in grads]).detach()


def approximate_influence(
    model, train_data, test_x, test_y, n_train_samples=500, device=device
):
    """
    Approximate influence function using gradient dot products.
    Full IF requires Hessian inversion — we use the simpler gradient similarity
    as a proxy (exact in the linear case, approximate for neural nets).

    Positive dot product → training example helps predict test correctly
    Negative dot product → training example hurts test prediction
    """
    test_grad = compute_grad(model, test_x, test_y, device)

    influences = []
    indices = np.random.choice(len(train_data), n_train_samples, replace=False)

    for idx in indices:
        x_tr, y_tr = train_data[idx]
        train_grad = compute_grad(model, x_tr, y_tr, device)
        # Gradient dot product as influence proxy
        influence = torch.dot(test_grad, train_grad).item()
        influences.append((idx, influence))

    return sorted(influences, key=lambda t: t[1], reverse=True)  # highest = most helpful


# Train a clean model, then find most influential training examples for a specific test case
clean_loader = DataLoader(train_data, batch_size=256, shuffle=True)
model_inf    = train_model(clean_loader, epochs=8)

# Find a specific test example to analyze
test_x, test_y = test_data[42]  # a specific test digit
print(f"Target test example: digit {test_y}")
print("Computing approximate influence scores for 500 training examples...")

influences = approximate_influence(model_inf, train_data, test_x, test_y, n_train_samples=500)

# Most helpful and most harmful training examples
top_helpful  = influences[:5]
top_harmful  = influences[-5:]

fig, axes = plt.subplots(2, 6, figsize=(14, 5))
fig.suptitle(f'Influence Analysis for Test Digit: {test_y}', fontsize=11)

def unnorm(t): return np.clip((t.squeeze().numpy() * 0.3081 + 0.1307), 0, 1)

axes[0, 0].imshow(unnorm(test_x), cmap='gray'); axes[0, 0].set_title('Test input', fontweight='bold'); axes[0, 0].axis('off')
axes[1, 0].axis('off')

for col, (idx, inf_val) in enumerate(top_helpful[:5]):
    x_tr, y_tr = train_data[idx]
    axes[0, col+1].imshow(unnorm(x_tr), cmap='gray')
    axes[0, col+1].set_title(f'Label:{y_tr}\nInf:+{inf_val:.2f}', fontsize=7, color='green')
    axes[0, col+1].axis('off')

for col, (idx, inf_val) in enumerate(top_harmful[:5]):
    x_tr, y_tr = train_data[idx]
    axes[1, col+1].imshow(unnorm(x_tr), cmap='gray')
    axes[1, col+1].set_title(f'Label:{y_tr}\nInf:{inf_val:.2f}', fontsize=7, color='red')
    axes[1, col+1].axis('off')

axes[0, 0].set_ylabel('Most helpful', fontsize=9)
axes[1, 0].text(0.5, 0.5, 'Most\nharmful', ha='center', va='center', fontsize=9, transform=axes[1,0].transAxes)

plt.tight_layout()
plt.savefig('influence_functions.png', bbox_inches='tight')
plt.show()
print("Helpful training examples are same-class, similar images.")
print("Harmful examples pull the model's decision boundary away from the test point.")

## 3. `poison-control`: Training Set Scanner

In [ ]:
def poison_control_scan(
    model: nn.Module,
    train_data,
    device,
    n_samples: int = 2000,
    contamination: float = 0.1,
) -> dict:
    """
    poison-control: training dataset poisoning scanner.

    Uses three complementary signals to flag suspicious training examples:
    1. High-loss examples: correct-class loss much higher than class median
    2. Activation outliers: penultimate-layer representation is anomalous within its class
    3. Label disagreement: model prediction disagrees with provided label (mislabeled?)

    Returns flagged indices and per-example suspicion scores.
    """
    model.eval()
    indices = np.random.choice(len(train_data), min(n_samples, len(train_data)), replace=False)
    subset  = torch.utils.data.Subset(train_data, indices)
    loader  = DataLoader(subset, batch_size=128)

    all_losses, all_labels, all_preds, all_acts = [], [], [], []

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            losses = F.cross_entropy(logits, y, reduction='none')

            # Activations: flatten conv output
            acts = F.relu(model.conv2(model.pool(F.relu(model.conv1(x)))))
            acts_flat = model.pool(acts).flatten(1)

            all_losses.append(losses.cpu().numpy())
            all_labels.append(y.cpu().numpy())
            all_preds.append(logits.argmax(1).cpu().numpy())
            all_acts.append(acts_flat.cpu().numpy())

    losses  = np.concatenate(all_losses)
    labels  = np.concatenate(all_labels)
    preds   = np.concatenate(all_preds)
    acts    = np.vstack(all_acts)

    # Signal 1: high loss relative to class median
    class_loss_medians = {c: np.median(losses[labels == c]) for c in range(10)}
    loss_z = np.array([losses[i] - class_loss_medians[labels[i]] for i in range(len(losses))])
    loss_score = (loss_z - loss_z.mean()) / (loss_z.std() + 1e-8)

    # Signal 2: activation outlier score (Isolation Forest per class)
    pca = PCA(n_components=20); acts_pca = pca.fit_transform(acts)
    iso = IsolationForest(contamination=contamination, random_state=42)
    outlier_scores = -iso.fit(acts_pca).score_samples(acts_pca)  # higher = more anomalous
    outlier_scores = (outlier_scores - outlier_scores.mean()) / (outlier_scores.std() + 1e-8)

    # Signal 3: label-prediction disagreement
    disagree_score = (preds != labels).astype(float)

    # Combined suspicion score
    suspicion = loss_score + outlier_scores + 2 * disagree_score

    # Flag top contamination fraction as suspicious
    threshold = np.percentile(suspicion, 100 * (1 - contamination))
    flagged_relative = np.where(suspicion >= threshold)[0]  # indices within our sample
    flagged_absolute = indices[flagged_relative]             # indices in original dataset

    report = {
        'n_scanned': len(indices),
        'n_flagged': len(flagged_absolute),
        'flag_rate': len(flagged_absolute) / len(indices),
        'flagged_indices': flagged_absolute,
        'label_disagreement_rate': disagree_score.mean(),
        'mean_loss': losses.mean(),
        'suspicion_scores': dict(zip(indices, suspicion)),
    }

    print(f"poison-control scan results:")
    print(f"  Examples scanned:           {report['n_scanned']:,}")
    print(f"  Examples flagged:           {report['n_flagged']:,} ({report['flag_rate']:.1%})")
    print(f"  Label disagreement rate:    {report['label_disagreement_rate']:.4f}")
    print(f"  Mean training loss:         {report['mean_loss']:.4f}")
    return report


# Scan clean dataset
model_clean_ref = train_model(DataLoader(train_data, batch_size=256, shuffle=True), epochs=8)
print("Scanning CLEAN training set:")
report_clean = poison_control_scan(model_clean_ref, train_data, device, n_samples=2000)

# Scan poisoned dataset
ds_flipped = LabelFlippedDataset(train_data, flip_rate=0.10)
model_poisoned_ref = train_model(DataLoader(ds_flipped, batch_size=256, shuffle=True), epochs=8)
print("\nScanning POISONED training set (10% label flips):")
report_poisoned = poison_control_scan(model_poisoned_ref, ds_flipped, device, n_samples=2000)

## Summary

```
Attack type         Detectability      Effectiveness   Access needed
─────────────────────────────────────────────────────────────────────
Random label flip   High (label review) High (at scale)  Label write
Targeted label flip High (label review) Moderate         Label write
Gradient-based      Low                 High             Model + data
Clean-label         Very low            Moderate         Model gradient
```

**`poison-control` scan signals:**
1. **High loss**: examples the model can't fit well are suspicious
2. **Activation outliers**: poisoned examples often have anomalous internal representations
3. **Label-prediction disagreement**: if the model disagrees with the label, that's a red flag

**Defenses:**
- **Label cleaning**: human review of high-suspicion examples flagged by `poison-control`
- **Data provenance**: track where each training example came from
- **Robust training**: use loss-clipped SGD or trimmed-mean gradients to reduce influence of outliers
- **Multiple sources**: cross-validate labels across independent annotators

**Next:** Notebook 11 covers supply chain attacks — what happens when the threat is not in your training data but in the model artifacts themselves (pickle exploits, ONNX graph injection, HuggingFace supply chain attacks).